# Healthcare Data Governance and Cleaning
**Purpose:** privacy-aware, reproducible quality checks for a small synthetic/de-identified training sample.

> Safety note: a `PatientID` that looks pseudonymous is not proof that data is anonymous. This sample is treated as training data based on the supplied task description; the file alone cannot prove its origin or guarantee that re-identification is impossible. Do not add names, addresses, phone numbers, or other identifiers. Do not use these records for clinical decisions.

## Rules
- Preserve source values and never invent clinical values.
- Trim surrounding whitespace and standardize category capitalization.
- Validate data types, missingness, ID uniqueness, selected technical ranges and blood-pressure format.
- Record exceptions; flag rather than silently remove rows or correct questionable clinical values.
- Confirm units, local policy and clinical ranges with an authorized data owner before production use.


In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

INPUT = Path("healthcare_patients_sample.csv")  # place source CSV beside this notebook
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

raw = pd.read_csv(INPUT)
df = raw.copy()
print("Rows, columns:", df.shape)
display(df.head())
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(),
                      "distinct_non_null": df.nunique(dropna=True)}))


In [ ]:
# Normalize text conservatively; no clinical-value imputation.
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

for col in ["Gender", "Diagnosis", "PrimaryMedication", "Readmitted"]:
    if col in df.columns:
        df[col] = df[col].map(lambda x: x.title() if isinstance(x, str) and x != "" else x)

if "Readmitted" in df.columns:
    df["Readmitted"] = df["Readmitted"].map(
        lambda x: {"yes": "Yes", "no": "No"}.get(x.lower(), x) if isinstance(x, str) else x
    )
if "BloodPressure" in df.columns:
    df["BloodPressure"] = df["BloodPressure"].map(
        lambda x: re.sub(r"\\s+", "", x) if isinstance(x, str) else x
    )


In [ ]:
exceptions = []
def flag(i, field, value, rule, action="Flagged; original value retained"):
    pid = df.at[i, "PatientID"] if "PatientID" in df.columns else ""
    exceptions.append({"source_row_number": int(i)+2, "PatientID": "" if pd.isna(pid) else pid,
                       "field": field, "observed_value": "" if pd.isna(value) else str(value),
                       "validation_rule": rule, "action_taken": action})

if "PatientID" in df.columns:
    for i, v in df["PatientID"].items():
        if pd.isna(v) or str(v).strip() == "":
            flag(i, "PatientID", v, "Must be present", "Flagged; ID not fabricated")
    for i in df.index[df["PatientID"].notna() & df["PatientID"].duplicated(keep=False)]:
        flag(i, "PatientID", df.at[i, "PatientID"], "PatientID should be unique",
             "Flagged; duplicate rows retained for review")

for c in df.columns:
    for i in df.index[df[c].isna()]:
        flag(i, c, np.nan, "Missing values must be assessed; do not invent clinical values",
             "Missing value preserved")

if "Age" in df.columns:
    for i,v in df["Age"].items():
        if pd.notna(v) and (not float(v).is_integer() or v < 0 or v > 120):
            flag(i,"Age",v,"Age must be whole number 0–120")
if "DosageMg" in df.columns:
    for i,v in df["DosageMg"].items():
        if pd.notna(v) and v < 0: flag(i,"DosageMg",v,"Dosage must not be negative")
if "TreatmentDurationDays" in df.columns:
    for i,v in df["TreatmentDurationDays"].items():
        if pd.notna(v) and (not float(v).is_integer() or v < 0):
            flag(i,"TreatmentDurationDays",v,"Duration must be a non-negative whole number")
if "CholesterolLevel" in df.columns:
    for i,v in df["CholesterolLevel"].items():
        if pd.notna(v) and (v < 0 or v > 1000):
            flag(i,"CholesterolLevel",v,"Broad technical check 0–1000; units and clinical range require review")
if "Gender" in df.columns:
    for i,v in df["Gender"].items():
        if pd.notna(v) and v not in {"Male","Female","Other","Unknown"}:
            flag(i,"Gender",v,"Unexpected category; confirm source vocabulary")
if "Readmitted" in df.columns:
    for i,v in df["Readmitted"].items():
        if pd.notna(v) and v not in {"Yes","No","Unknown"}:
            flag(i,"Readmitted",v,"Expected Yes/No/Unknown")
if "BloodPressure" in df.columns:
    pattern = re.compile(r"^\\d{2,3}/\\d{2,3}$")
    for i,v in df["BloodPressure"].items():
        if pd.notna(v) and not pattern.match(str(v)):
            flag(i,"BloodPressure",v,"Expected systolic/diastolic numeric format such as 120/80")

for i in df.index[df.duplicated(keep=False)]:
    flag(i,"__row__","exact duplicate row","Review exact duplicate rows",
         "Flagged; row retained for review")

pd.DataFrame(exceptions, columns=["source_row_number","PatientID","field","observed_value",
                                  "validation_rule","action_taken"]).to_csv(OUT/"exception_log.csv",index=False)
df.to_csv(OUT/"healthcare_patients_cleaned.csv",index=False)
print("Cleaned file and exception log saved in outputs/")


In [ ]:
# Final quality summary
summary = pd.DataFrame({
    "metric": ["row_count", "column_count", "missing_cells", "duplicate_patient_id_rows", "exact_duplicate_rows"],
    "value": [
        len(df), len(df.columns), int(df.isna().sum().sum()),
        int(df["PatientID"].duplicated(keep=False).sum()) if "PatientID" in df.columns else "N/A",
        int(df.duplicated(keep=False).sum())
    ]
})
display(summary)
display(pd.DataFrame(exceptions).head(20))
